Yeh example [is blog post](https://medium.com/mlreview/making-ai-art-with-style-transfer-using-keras-8bb5fa44b216) se inspired hai, aur iska kaafi code wahin se liya gaya hai. CNTK framework use karke style transfer ka ek aur achha example [yahan](https://github.com/Microsoft/CNTK/blob/master/Tutorials/CNTK_205_Artistic_Style_Transfer.ipynb) available hai. [Artistic Style Transfer](https://arxiv.org/abs/1508.06576) par original research paper yahan hai.

Style transfer ke peeche ke main ideas yeh hain:

* White noise se shuru karke, hum current image $x$ ko optimize karne ki koshish karte hain taaki ek loss function minimize ho.
* Loss function teen components se bana hota hai $\mathcal{L(x)} = \alpha\mathcal{L}_c(x,i) + \beta\mathcal{L}_s(x,s)+\gamma\mathcal{L}_t(x)$
   - $\mathcal{L}_c$ - content loss - yeh dikhata hai ki current image $x$ original image $i$ ke kitne kareeb hai.
   - $\mathcal{L}_s$ - style loss - yeh dikhata hai ki current image $x$ style image $s$ ke kitne kareeb hai.
   - $\mathcal{L}_t$ - total variation loss (hum ise apne example mein consider nahi karenge) - yeh ensure karta hai ki final image smooth ho, yaani yeh image $x$ ke neighboring pixels ka mean squared error dikhata hai.

In loss functions ko is tarah design karna hota hai ki, jaise, style loss images ki style similarity ko reflect kare, na ki actual content ko. Iske liye, hum ek CNN ke kuch deep feature layers ko compare karenge, jo image ko dekhta hai.

Chaliye kuch images load karke shuru karte hain:

In [ ]:
!curl https://cdn.pixabay.com/photo/2016/05/18/00/27/franz-marc-1399594_960_720.jpg > images/style.jpg
!curl https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_tabby_and_white_kitten_n01.jpg/1280px-Golden_tabby_and_white_kitten_n01.jpg > images/image.jpg

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fmin_l_bfgs_b

In [ ]:
img_size = 256

def load_image(fn):
  x = cv2.imread(fn)
  return cv2.cvtColor(x, cv2.COLOR_BGR2RGB)

img_style = load_image('images/style.jpg')
img_content = load_image('images/image.jpg')
img_content = img_content[:,200:200+857,:]
img_content = cv2.resize(img_content,(img_size,img_size))
img_style = img_style[:,200:200+671,:]
img_style = cv2.resize(img_style,(img_size,img_size))

img_result = np.random.randint(256,size=(img_size,img_size,3)).astype(np.float64)

fig,ax = plt.subplots(1,3)
ax[0].imshow(img_content)
ax[1].imshow(img_style)
ax[2].imshow(img_result.astype(np.int))
plt.show()

In [ ]:
import tensorflow as tf
from tf.keras import backend as K
from tf.keras.applications.vgg16 import preprocess_input
from tf.keras.applications import VGG16
from tf.keras.preprocessing.image import load_img, img_to_array
tf_session = K.get_session()

In [ ]:
img_content_var = K.variable(preprocess_input(np.expand_dims(img_content, axis=0)), dtype='float32')
img_style_var = K.variable(preprocess_input(np.expand_dims(img_style, axis=0)), dtype='float32')
img_result_1 = preprocess_input(np.expand_dims(img_result, axis=0))
img_result_holder = K.placeholder(shape=(1, img_size, img_size, 3))

Style loss aur content loss calculate karne ke liye, humein ek CNN dwara nikaale gaye feature space mein kaam karna hoga. Hum alag-alag CNN architectures use kar sakte hain, lekin hamare case mein simplicity ke liye hum *ImageNet* par pre-trained VGG-16 chunenge.

In [ ]:
cModel = VGG16(include_top=False, weights='imagenet', input_tensor=img_content_var)
sModel = VGG16(include_top=False, weights='imagenet', input_tensor=img_style_var)
gModel = VGG16(include_top=False, weights='imagenet', input_tensor=img_result_holder)

In [ ]:
gModel.summary()

## Content loss

**Content loss** yeh dikhayega ki hamari current image $x$ original image ke kitne kareeb hai. Yeh CNN mein intermediate feature layers ko dekhta hai aur squared error calculate karta hai. Layer $l$ par content loss ko is tarah define kiya jayega:
$$
\mathcal{L}_c = {1\over2}\sum_{i,j} (F_{ij}^{(l)}-P_{ij}^{(l)})^2
$$
jahan $F^{(l)}$ aur $P^{(l)}$ -- layer $l$ par features hain.

In [ ]:
def get_feature_reps(x, layer_names, model):
    featMatrices = []
    for ln in layer_names:
        selectedLayer = model.get_layer(ln)
        featRaw = selectedLayer.output
        featRawShape = K.shape(featRaw).eval(session=tf_session)
        N_l = featRawShape[-1]
        M_l = featRawShape[1]*featRawShape[2]
        featMatrix = K.reshape(featRaw, (M_l, N_l))
        featMatrix = K.transpose(featMatrix)
        featMatrices.append(featMatrix)
    return featMatrices

def get_content_loss(F, P):
    cLoss = 0.5*K.sum(K.square(F - P))
    return cLoss

Chaliye dekhte hain ki alag-alag layers ke features image ko kaise affect karte hain. Aisa karne ke liye, hum sirf ek layer ke liye content loss ko minimize karne ki koshish karenge. Iske liye hum SciPy ka `fmin_l_bfgs_b` function use karenge, jo ek function ko minimize karne aur uska gradient lene ka kaam karta hai (hamare case mein, `helper_loss` function loss function aur gradient dono return karta hai).

**Important**: Hamare case mein, saari calculations GPU-aware *TensorFlow* framework use karke ki jaati hain. `helper_loss` function ek computational graph return karta hai, jise kisi di gayi image ke liye loss calculate karne ke liye use kiya ja sakta hai, aur yeh `K.gradients` use karke automatically gradient calculate karta hai.

In [ ]:
layer='block4_conv2'
P = get_feature_reps(x=img_content_var, layer_names=[layer], model=cModel)[0]
x = img_result.flatten()
def helper_loss(img):
    R = get_feature_reps(img,[layer],gModel)[0]
    return get_content_loss(R,P)
  
def help_loss(x):
    if x.shape != (1, img_size, img_size, 3):
        x = x.reshape((1,img_size, img_size, 3))
    keras_fcn = K.function([gModel.input], [helper_loss(gModel.input)])
    keras_grad = K.function([gModel.input], K.gradients(helper_loss(gModel.input),[gModel.input]))
    return keras_fcn([x])[0].astype('float64'),keras_grad([x])[0].flatten().astype('float64')

x, _, _ = fmin_l_bfgs_b(help_loss, x, maxiter=30, disp=True)

In [ ]:
def postprocess_array(x):
    # Zero-center by mean pixel
    if x.shape != (img_size, img_size, 3):
        x = x.reshape((img_size, img_size, 3))
    x[..., 0] += 103.939
    x[..., 1] += 116.779
    x[..., 2] += 123.68
    # 'BGR'->'RGB'
    x = x[..., ::-1]
    x = np.clip(x, 0, 255)
    x = x.astype('uint8')
    return x


In [ ]:
plt.imshow(postprocess_array(x.copy()))
plt.show()

In [ ]:
layer='block3_conv2'
P = get_feature_reps(x=img_content_var, layer_names=[layer], model=cModel)[0]
x = img_result.flatten()
x, _, _ = fmin_l_bfgs_b(help_loss, x, maxiter=30, disp=True)
plt.imshow(postprocess_array(x.copy()))
plt.show()

In [ ]:
layer='block5_conv1'
P = get_feature_reps(x=img_content_var, layer_names=[layer], model=cModel)[0]
x = img_result.flatten()
x, _, _ = fmin_l_bfgs_b(help_loss, x, maxiter=30, disp=True)
plt.imshow(postprocess_array(x.copy()))
plt.show()

## Style loss

Style loss ek aisa measure hai jo batata hai ki ek image ki style ko kitni achhi tarah se dobara banaya gaya hai. Yeh aam taur par deep learning models mein use hota hai, khaas taur par style transfer tasks mein.

### Style loss kaise calculate hota hai?

Style loss calculate karne ke liye, hum aam taur par "Gram matrix" use karte hain. Gram matrix ek particular layer ke feature maps ke beech correlation ko measure karta hai. Yeh style ki information ko capture karne mein madad karta hai.

#### Steps:

1. **Feature maps nikaalein**
   Model ki ek layer se feature maps lijiye. Yeh maps image ki alag-alag features ko represent karte hain.

2. **Gram matrix calculate karein**
   Feature maps ke correlation ko measure karne ke liye Gram matrix use kijiye. Yeh style ki information ko capture karta hai.

3. **Target style se compare karein**
   Calculate kiye gaye Gram matrix ko target style ke Gram matrix se compare kijiye. Difference jitna kam hoga, style utni hi behtar tarah se reconstruct hogi.

### Style loss kyun use karein?

Style loss isliye use hota hai taaki yeh ensure ho sake ki output image mein desired style ki features shaamil hon. Yeh khaas taur par tab useful hota hai jab aap ek image ke content ko doosri image ki style ke saath combine karna chahte hain.

### Common challenges

- **Balance banaye rakhna**
   Style aur content loss ke beech sahi balance banana challenging ho sakta hai. Agar style loss bahut zyada hai, toh output image mein content ki information kho sakti hai.

- **Computation ki complexity**
   Gram matrix ki calculation expensive ho sakti hai, khaas taur par high-resolution images ke liye.

### Conclusion

Style loss deep learning models mein ek important component hai, khaas taur par style transfer tasks mein. Yeh ensure karta hai ki output image desired style ko effectively capture kare.

Style loss style transfer ke peeche ka main concept hai. Hum actual features ko compare nahi karte, balki unke Gram matrices ko compare karte hain, jo is tarah define hote hain: $$G=A\times A^T$$

Gram matrix correlation matrix jaisa hota hai, aur yeh dikhata hai ki kuch filters doosre filters par kaise depend karte hain. Style loss ko alag-alag layers se aane wale losses ke sum ke roop mein calculate kiya jaata hai, jinhe aksar weighted coefficients ke saath consider kiya jaata hai.

Style transfer ke liye total loss function *content loss* aur *style loss* ka sum hota hai.

In [ ]:
def get_Gram_matrix(F):
    G = K.dot(F, K.transpose(F))
    return G

def get_style_loss(ws, Gs, As):
    sLoss = K.variable(0.)
    for w, G, A in zip(ws, Gs, As):
        M_l = K.int_shape(G)[1]
        N_l = K.int_shape(G)[0]
        G_gram = get_Gram_matrix(G)
        A_gram = get_Gram_matrix(A)
        sLoss+= w*0.25*K.sum(K.square(G_gram - A_gram))/ (N_l**2 * M_l**2)
    return sLoss
  
def get_total_loss(gImPlaceholder, alpha=1.0, beta=30.0):
    F = get_feature_reps(gImPlaceholder, layer_names=[content_layer_name], model=gModel)[0]
    Gs = get_feature_reps(gImPlaceholder, layer_names=style_layer_names, model=gModel)
    contentLoss = get_content_loss(F, P)
    styleLoss = get_style_loss(ws, Gs, As)
    totalLoss = alpha*contentLoss + beta*styleLoss
    return totalLoss
 

## Sab kuch ek saath jodna

Yahan `calualate_loss` function total loss calculate karega:

In [ ]:
def calculate_loss(gImArr):
  """
  Calculate total loss using K.function
  """
  if gImArr.shape != (1, img_size, img_size, 3):
        gImArr = gImArr.reshape((1,img_size, img_size, 3))
  loss_fcn = K.function([gModel.input], [get_total_loss(gModel.input)])
  grad_fcn = K.function([gModel.input], 
                        K.gradients(get_total_loss(gModel.input), [gModel.input]))
  return loss_fcn([gImArr])[0].astype('float64'),grad_fcn([gImArr])[0].flatten().astype('float64')

content_layer_name = 'block4_conv2'
style_layer_names = ['block1_conv1','block2_conv1','block3_conv1','block4_conv1']

P = get_feature_reps(x=img_content_var, layer_names=[content_layer_name], model=cModel)[0]
As = get_feature_reps(x=img_style_var, layer_names=style_layer_names, model=sModel)
ws = np.ones(len(style_layer_names))/float(len(style_layer_names))

img_result = np.random.randint(256,size=(img_size,img_size,3)).astype(np.float64)
img_result_1 = preprocess_input(np.expand_dims(img_result, axis=0))

iterations = 10
x_opt = img_result_1.flatten()

Neeche diya gaya code loss ka actual optimization karta hai. Dhyan rakhiye ki GPU ke saath bhi optimization mein kaafi time lagta hai. Aap results sudharne ke liye neeche wale cell ko kai baar run kar sakte hain.

In [ ]:
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, x_opt, maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()

In [ ]:
iterations = 20
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, xopt, fprime=get_grad,
                            maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()

## Variation loss add karein

**Variation loss** humein image ko kam noisy banane deta hai, neighboring pixels ke beech ka difference kam karke.

In [ ]:
def total_variation_loss(x):
    a = K.square(x[:,:img_size-1,:img_size-1,:] - x[:, 1:, :img_size-1,:])
    b = K.square(x[:,:img_size-1,:img_size-1,:] - x[:, :img_size-1, 1:,:])
    return K.sum(K.pow(a + b, 1.25))
  
def get_total_loss(gImPlaceholder, alpha=1.0, beta=30.0):
    F = get_feature_reps(gImPlaceholder, layer_names=[content_layer_name], model=gModel)[0]
    Gs = get_feature_reps(gImPlaceholder, layer_names=style_layer_names, model=gModel)
    contentLoss = get_content_loss(F, P)
    styleLoss = get_style_loss(ws, Gs, As)
    variationLoss = total_variation_loss(gImPlaceholder)
    totalLoss = alpha*contentLoss + beta*styleLoss + variationLoss
    return totalLoss

img_result = np.random.randint(256,size=(img_size,img_size,3)).astype(np.float64)
img_result_1 = preprocess_input(np.expand_dims(img_result, axis=0))
iterations = 10
x_opt = img_result_1.flatten()


In [ ]:
xopt, f_val, info= fmin_l_bfgs_b(calculate_loss, x_opt, maxiter=iterations, disp=True)
plt.imshow(postprocess_array(xopt.copy()))
plt.show()

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->